# AutoSolve Standalone ML Training Pipeline 🚀

Welcome! This notebook is a **completely self-contained training suite** for the AutoSolve Blender camera tracking addon. It contains all feature extraction, PyTorch training loops, and ONNX/NumPy exporters inside the cells below.

---

## 📖 Easy 3-Step Google Colab Flow:

### 1. Upload Your Videos
* In your **Google Drive**, create a folder named `AutoSolve_ML_Data` and a subfolder named `clips` inside it (i.e. `AutoSolve_ML_Data/clips`).
* Upload all your video clips (.mp4, .mkv, .mov, etc.) into the `clips` folder.

### 2. Run All Cells
* In the menu above, click **Runtime → Run all** (or run each cell sequentially from top to bottom by pressing `Shift + Enter`).
* When prompted in the second cell, authorize Google Drive access.

### 3. Copy Generated Models to Blender Addon
Once training completes, download the exported files from the Colab file browser and copy them into your local Blender addon folders:

* **ONNX & NumPy Models:** Copy `settings_model.onnx`, `track_predictor.onnx`, `patch_rigidity.onnx`, `track_predictor_meta.json`, `settings_model_meta.json`, `patch_rigidity_meta.json`, and `track_predictor.json` to:
  📂 `autosolve/tracker/models/`

* **Presets:** Copy `region_weights.json` and `defaults.json` to:
  📂 `autosolve/tracker/presets/`

---


In [ ]:
#@title Configure Google Drive Integration
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

USE_GOOGLE_DRIVE = True #@param {type:"boolean"}
DRIVE_PROJECT_PATH = "AutoSolve_ML_Data" #@param {type:"string"}

import os
import shutil

in_colab = False
try:
    import google.colab
    in_colab = True
except ImportError:
    pass

if in_colab and USE_GOOGLE_DRIVE:
    print("Mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Define project directory on Drive
    persist_dir = f"/content/drive/MyDrive/{DRIVE_PROJECT_PATH}"
    os.makedirs(persist_dir, exist_ok=True)
    
    # Define directories inside persistent storage
    CLIPS_DIR = os.path.join(persist_dir, "clips")
    RAW_DIR = os.path.join(persist_dir, "data", "raw")
    PROCESSED_DIR = os.path.join(persist_dir, "data", "processed")
    RUNS_DIR = os.path.join(persist_dir, "runs")
    ONNX_OUT_DIR = os.path.join(RUNS_DIR, "onnx")
else:
    print("Using local runtime filesystem.")
    base_dir = "/content/AutoSolve_ML_Data" if in_colab else "./AutoSolve_ML_Data"
    CLIPS_DIR = os.path.join(base_dir, "clips")
    RAW_DIR = os.path.join(base_dir, "data", "raw")
    PROCESSED_DIR = os.path.join(base_dir, "data", "processed")
    RUNS_DIR = os.path.join(base_dir, "runs")
    ONNX_OUT_DIR = os.path.join(RUNS_DIR, "onnx")

# Create all directories
for d in [CLIPS_DIR, RAW_DIR, PROCESSED_DIR, RUNS_DIR, ONNX_OUT_DIR]:
    os.makedirs(d, exist_ok=True)

print("\n📂 Active Workspace Directories:")
print(f"   Clips Input:      {CLIPS_DIR}")
print(f"   Raw Solve Logs:   {RAW_DIR}")
print(f"   Processed Data:   {PROCESSED_DIR}")
print(f"   Runs & Checkpts:  {RUNS_DIR}")
print(f"   ONNX Export Dir:  {ONNX_OUT_DIR}")


In [ ]:
# Install dependencies
# Note: Install 'ultralytics' for YOLOv8 support. Meta's CoTracker v3 requires GPU and is optional.
!pip install -q torch numpy onnx onnxruntime opencv-python matplotlib ultralytics

!pip install -q git+https://github.com/facebookresearch/co-tracker.git

import json
import torch
import numpy as np
import onnx
import onnxruntime as ort
import cv2
import matplotlib.pyplot as plt
import random
import math
from typing import List, Tuple, Dict, Any

print(f"PyTorch:     {torch.__version__}")
print(f"NumPy:       {np.__version__}")
print(f"ONNX:        {onnx.__version__}")
print(f"onnxruntime: {ort.__version__}")
print(f"GPU (CUDA):  {torch.cuda.is_available()}")


In [ ]:
# GPU availability guard
import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected! Go to Runtime -> Change runtime type -> T4 GPU and reconnect."
    )
print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:
# Constants and Metadata Definitions
FOOTAGE_TYPES = ['AUTO', 'INDOOR', 'OUTDOOR', 'DRONE', 'HANDHELD', 'GIMBAL', 'ACTION', 'VFX', 'SCREEN', 'CINEMATIC']
MOTION_MODELS = ['Loc', 'LocRot', 'Affine', 'Perspective']
RESOLUTION_CLASSES = ['HD_24fps', 'HD_30fps', 'HD_60fps', '4K_24fps', '4K_30fps']

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def get_one_hot(value: str, catalog: List[str]) -> List[float]:
    one_hot = [0.0] * len(catalog)
    if value in catalog:
        one_hot[catalog.index(value)] = 1.0
    else:
        one_hot[0] = 1.0
    return one_hot

def classify_region(fx: float, fy: float) -> str:
    ry = "top" if fy > 0.66 else ("bottom" if fy < 0.33 else "mid")
    rx = "left" if fx < 0.33 else ("right" if fx > 0.66 else "center")
    return "center" if (ry == "mid" and rx == "center") else (f"{ry}-{rx}" if ry != "mid" else f"mid-{rx}")

def extract_sample_features(sample: Dict[str, Any], video_meta: Dict[str, float] = None) -> List[float]:
    meta = sample["clip_metadata"]
    settings = sample["settings"]
    
    # 1. Clip features (4)
    clip_feats = [
        float(meta.get("width", 1920)),
        float(meta.get("height", 1080)),
        float(meta.get("fps", 24.0)),
        float(meta.get("frame_count", 250))
    ]
    
    # 2. Boolean switches (2)
    switches = [
        1.0 if settings.get("tripod_mode", False) else 0.0,
        1.0 if settings.get("robust_mode", False) else 0.0
    ]
    
    # 3. Settings features (4)
    setting_feats = [
        float(settings.get("pattern_size", 17)),
        float(settings.get("search_size", 71)),
        float(settings.get("correlation", 0.70)),
        float(settings.get("threshold", 0.30))
    ]
    
    # 4. One-hots (14)
    f_type_oh = get_one_hot(settings.get("footage_type", "AUTO"), FOOTAGE_TYPES)
    m_model_oh = get_one_hot(settings.get("motion_model", "LocRot"), MOTION_MODELS)
    
    # 5. Video features (4)
    if video_meta is None:
        video_meta = {}
    v_feats = [
        float(video_meta.get("mean_motion", 0.5)),
        float(video_meta.get("zoom_divergence", 0.0)),
        float(video_meta.get("distortion_factor", 0.0)),
        float(video_meta.get("noise_ratio", 0.003))
    ]
    
    return clip_feats + switches + setting_feats + f_type_oh + m_model_oh + v_feats

def calculate_reward(sample: Dict[str, Any]) -> float:
    success = 1.0 if sample.get("solve_success", False) else 0.0
    error = sample.get("solve_error", 10.0)
    bundle_ratio = sample.get("bundle_ratio", 0.0)
    
    # Clamp error contribution
    error_penalty = max(0.0, min(1.0, error / 5.0))
    reward = success * (1.0 - error_penalty) * bundle_ratio
    return round(reward, 4)


In [ ]:
def compute_dynamic_pixel_ratio(semantic_fp: str) -> float | None:
    """Returns mean dynamic object area ratio across frames, or None on failure."""
    try:
        with open(semantic_fp, 'r', encoding='utf-8') as f:
            sem_d = json.load(f)
        frames = sem_d.get("frames", [])
        if not frames:
            return None
        total = 0.0
        for frame in frames:
            area = 0.0
            for det in frame.get("detections", []):
                if det.get("is_dynamic", False):
                    box = det.get("box", [0, 0, 0, 0])
                    area += max(0.0, box[2] - box[0]) * max(0.0, box[3] - box[1])
            total += min(1.0, area)
        return round(total / len(frames), 4)
    except Exception as e:
        print(f"Warning: dynamic ratio calculation failed ({e})")
        return None

SKIP_SUFFIXES = {
    '_video_meta.json', 'settings_dataset.json', 'recommended_defaults.json',
    'track_predictor.json', '_semantic_meta.json', 'defaults.json', '_base_trajectories.json'
}

def is_solve_json(filename: str) -> bool:
    return filename.endswith('.json') and not any(filename.endswith(s) for s in SKIP_SUFFIXES)

# Ingestion, Feature Extraction, and Tracking Pipeline Functions
import json
import os
import shutil

_cotracker_predictor_cache = None
_yolo_model_cache = None

def extract_features_from_video(video_path: str) -> dict:
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video file: {video_path}")

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 24.0
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    step = max(1, frame_count // 30)
    prev_gray = None
    motions, divergences, noises, curvatures = [], [], [], []
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret: break

        if frame_idx % step == 0:
            small_frame = cv2.resize(frame, (320, 180))
            gray = cv2.cvtColor(small_frame, cv2.COLOR_BGR2GRAY)

            # Grain Noise
            blurred = cv2.GaussianBlur(gray, (5, 5), 0)
            high_freq = cv2.absdiff(gray, blurred)
            noises.append(float(np.mean(high_freq)))

            # Line Curvature
            edges = cv2.Canny(gray, 50, 150)
            lines = cv2.HoughLinesP(edges, 1, np.pi/180, 50, minLineLength=30, maxLineGap=10)
            if lines is not None:
                angles = [np.arctan2(l[0][3] - l[0][1], l[0][2] - l[0][0]) * 180 / np.pi for l in lines]
                curvatures.append(float(np.var(angles)) if angles else 0.0)
            else:
                curvatures.append(0.0)

            # Dense Flow
            if prev_gray is not None:
                flow = cv2.calcOpticalFlowFarneback(prev_gray, gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
                fx, fy = flow[..., 0], flow[..., 1]
                motions.append(float(np.mean(np.sqrt(fx**2 + fy**2))))
                div = np.gradient(fx, axis=1) + np.gradient(fy, axis=0)
                divergences.append(float(np.mean(div)))
            prev_gray = gray
        frame_idx += 1

    cap.release()
    return {
        "clip_name": os.path.splitext(os.path.basename(video_path))[0],
        "width": width, "height": height, "fps": fps, "frame_count": frame_count,
        "mean_motion": float(np.mean(motions)) if motions else 0.0,
        "zoom_divergence": float(np.mean(np.abs(divergences))) if divergences else 0.0,
        "distortion_factor": float(np.mean(curvatures)) if curvatures else 0.0,
        "grain_noise": float(np.mean(noises)) if noises else 0.0,
        "noise_ratio": float(np.mean(noises)) if noises else 0.0
    }

def run_opencv_tracking(video_path: str, grid_size: int = 8) -> Tuple[List[List[Tuple[float, float]]], dict]:
    cap = cv2.VideoCapture(video_path)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 24.0
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    ret, frame = cap.read()
    if not ret:
        cap.release()
        raise RuntimeError(f"Could not read first frame of {video_path}")
        
    gray_prev = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    pts = cv2.goodFeaturesToTrack(gray_prev, maxCorners=grid_size*grid_size, qualityLevel=0.01, minDistance=20)
    if pts is None:
        pts = np.array([[[x, y]] for y in np.linspace(height*0.1, height*0.9, grid_size) for x in np.linspace(width*0.1, width*0.9, grid_size)], dtype=np.float32)

    num_tracks = len(pts)
    trajectories = [[(float(p[0][0]/width), float(p[0][1]/height))] for p in pts]
    active = np.ones(num_tracks, dtype=bool)
    p_prev = pts.copy()

    while True:
        ret, frame = cap.read()
        if not ret: break
        gray_curr = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        p_curr, status, _ = cv2.calcOpticalFlowPyrLK(gray_prev, gray_curr, p_prev, None, winSize=(21, 21), maxLevel=3)

        for idx in range(num_tracks):
            if not active[idx]: continue
            if status[idx][0] == 0:
                active[idx] = False
                continue
            x, y = p_curr[idx][0]
            if x < 0 or x >= width or y < 0 or y >= height:
                active[idx] = False
                continue
            trajectories[idx].append((float(x/width), float(y/height)))
        p_prev = p_curr
        gray_prev = gray_curr
    cap.release()
    return trajectories, {"clip_name": os.path.splitext(os.path.basename(video_path))[0], "width": width, "height": height, "fps": fps, "frame_count": frame_count}

def run_cotracker_tracking(video_path: str, grid_size: int = 8) -> Tuple[List[List[Tuple[float, float]]], dict]:
    global _cotracker_predictor_cache
    try:
        from cotracker.predictor import CoTrackerPredictor
    except ImportError:
        print("CoTracker package not installed. Falling back to OpenCV tracking.")
        return run_opencv_tracking(video_path, grid_size)

    checkpoint_dir = os.path.join(RUNS_DIR, "checkpoints")
    checkpoint_path = os.path.join(checkpoint_dir, "scaled_offline.pth")
    if not os.path.exists(checkpoint_path):
        print("Downloading CoTracker v3 offline checkpoint from Hugging Face...")
        os.makedirs(checkpoint_dir, exist_ok=True)
        import urllib.request
        url = "https://huggingface.co/facebook/cotracker3/resolve/main/scaled_offline.pth"
        try:
            urllib.request.urlretrieve(url, checkpoint_path)
            print("Download complete.")
        except Exception as e:
            print(f"Failed to download checkpoint: {e}. Falling back to OpenCV tracking.")
            return run_opencv_tracking(video_path, grid_size)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    if _cotracker_predictor_cache is not None:
        model = _cotracker_predictor_cache
    else:
        print(f"Loading CoTracker v3 predictor on {device}...")
        try:
            model = CoTrackerPredictor(checkpoint=checkpoint_path).to(device)
            model.eval()
            _cotracker_predictor_cache = model
        except Exception as e:
            print(f"Failed to load CoTracker: {e}. Falling back to OpenCV tracking.")
            return run_opencv_tracking(video_path, grid_size)
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video file: {video_path}")

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 24.0
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    target_h, target_w = 288, 384
    scale = min(target_w / width, target_h / height)
    new_w = int(width * scale)
    new_h = int(height * scale)
    pad_w = (target_w - new_w) // 2
    pad_h = (target_h - new_h) // 2

    frames = []
    while True:
        ret, frame = cap.read()
        if not ret: break
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame_resized = cv2.resize(frame_rgb, (new_w, new_h))
        padded_frame = np.zeros((target_h, target_w, 3), dtype=np.uint8)
        padded_frame[pad_h:pad_h+new_h, pad_w:pad_w+new_w] = frame_resized
        frames.append(padded_frame)
    cap.release()

    if not frames:
        raise RuntimeError(f"Could not read frames from {video_path}")

    # Subsample frames to reduce VRAM usage
    FRAME_SKIP = 2
    frames = frames[::FRAME_SKIP]

    video_tensor = torch.from_numpy(np.stack(frames, axis=0))
    video_tensor = video_tensor.float()
    video_tensor = video_tensor.permute(0, 3, 1, 2).unsqueeze(0).to(device)

    print(f"Running CoTracker v3 inference (grid size {grid_size}x{grid_size})...")
    try:
        with torch.no_grad():
            if device == "cuda":
                with torch.amp.autocast(device_type="cuda"):
                    pred_tracks, pred_visibility = model(video_tensor, grid_size=grid_size)
            else:
                pred_tracks, pred_visibility = model(video_tensor, grid_size=grid_size)
    except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
        if device == "cuda" and ("out of memory" in str(e).lower() or isinstance(e, torch.cuda.OutOfMemoryError)):
            torch.cuda.empty_cache()
            print(f"  CoTracker OOM on {os.path.basename(video_path)}, falling back to OpenCV tracking.")
            return run_opencv_tracking(video_path, grid_size)
        else:
            raise e

    pred_tracks = pred_tracks.cpu().numpy()[0]
    pred_visibility = pred_visibility.cpu().numpy()[0]
    
    num_tracks = pred_tracks.shape[1]
    trajectories = [[] for _ in range(num_tracks)]
    
    for t_idx in range(num_tracks):
        for f_idx in range(len(frames)):
            if pred_visibility[f_idx, t_idx]:
                px, py = pred_tracks[f_idx, t_idx]
                nx = float((px - pad_w) / new_w)
                ny = float((py - pad_h) / new_h)
                nx = max(0.0, min(1.0, nx))
                ny = max(0.0, min(1.0, ny))
                trajectories[t_idx].append((nx, ny))
                
    meta = {"clip_name": os.path.splitext(os.path.basename(video_path))[0], "width": width, "height": height, "fps": fps, "frame_count": frame_count}
    return trajectories, meta

def run_yolo_semantic_masking(video_path: str, out_path: str) -> bool:
    global _yolo_model_cache
    try:
        from ultralytics import YOLO
    except ImportError:
        print("Ultralytics package not installed. Skipping YOLOv8 semantic features.")
        return False

    device = "cuda" if torch.cuda.is_available() else "cpu"
    if _yolo_model_cache is not None:
        model = _yolo_model_cache
    else:
        print(f"Loading YOLOv8 segmenter on {device}...")
        try:
            model = YOLO("yolov8n-seg.pt").to(device)
            _yolo_model_cache = model
        except Exception as e:
            print(f"Failed to load YOLO model: {e}. Skipping semantic features.")
            return False

    print(f"Running YOLOv8 segmentation on {os.path.basename(video_path)} ({device})...")
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error opening video: {video_path}")
        return False

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    clip_name = os.path.splitext(os.path.basename(video_path))[0]

    DYNAMIC_CLASSES = {
        'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train',
        'truck', 'boat', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow',
        'elephant', 'bear', 'zebra', 'giraffe'
    }

    frames_metadata = []
    frame_idx = 0
    with torch.inference_mode():
        while True:
            ret, frame = cap.read()
            if not ret: break

            detections = []
            results = model(frame, verbose=False)[0]
            if results.boxes is not None:
                for box in results.boxes:
                    cls_id = int(box.cls[0].item())
                    cls_name = results.names[cls_id]
                    conf = float(box.conf[0].item())
                    xyxyn = box.xyxyn[0].cpu().numpy().tolist()
                
                    is_dynamic = cls_name in DYNAMIC_CLASSES
                    detections.append({
                        "class": cls_name,
                        "confidence": round(conf, 3),
                        "box": [round(coord, 4) for coord in xyxyn],
                        "is_dynamic": is_dynamic
                    })
            frames_metadata.append({
                "frame_idx": frame_idx,
                "detections": detections
            })
            frame_idx += 1
        
    cap.release()
    
    output_data = {
        "clip_name": clip_name,
        "width": width,
        "height": height,
        "frame_count": frame_count,
        "frames": frames_metadata
    }
    
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(output_data, f, indent=4)
    print(f"Saved YOLOv8 semantic mask data to {out_path}")
    return True

def simulate_variation(
    base_trajectories: List[List[Tuple[float, float]]],
    meta: Dict[str, Any],
    quality: str,
    robust: bool,
    tripod: bool
) -> Dict[str, Any]:
    random.seed(hash(f"{meta['clip_name']}_{quality}_{robust}_{tripod}") % 1234567)

    # Adjust parameters based on tracking preset
    if quality == "QUALITY":
        noise_std = 0.001
        survival_base_prob = 0.85
    elif quality == "FAST":
        noise_std = 0.005
        survival_base_prob = 0.50
    else:  # BALANCED
        noise_std = 0.002
        survival_base_prob = 0.70

    if robust:
        survival_base_prob += 0.15
        noise_std *= 0.8

    survival_base_prob = min(0.95, max(0.20, survival_base_prob))
    track_samples = []

    for idx_t, base_coords in enumerate(base_trajectories):
        if len(base_coords) < 6:
            continue

        fx, fy = base_coords[0]
        region = classify_region(fx, fy)

        # Region-based survival multiplier
        region_survival_mult = 1.0
        if "top" in region:
            region_survival_mult *= 0.7
        if "left" in region or "right" in region:
            region_survival_mult *= 0.85
        if "center" in region:
            region_survival_mult *= 1.1

        survival_prob = min(0.98, survival_base_prob * region_survival_mult)
        survived = random.random() < survival_prob
        has_bundle = survived and (random.random() < 0.90)

        coords = []
        if survived:
            for x, y in base_coords:
                nx = x + random.normalvariate(0.0, noise_std)
                ny = y + random.normalvariate(0.0, noise_std)
                coords.append((max(0.0, min(1.0, nx)), max(0.0, min(1.0, ny))))
            average_error = random.uniform(0.15, 0.45)
        else:
            fail_frame = random.randint(5, len(base_coords) - 1)
            slip_frames = min(5, fail_frame)
            for f_idx in range(fail_frame):
                x, y = base_coords[f_idx]
                if f_idx >= fail_frame - slip_frames:
                    drift_factor = (f_idx - (fail_frame - slip_frames) + 1) * 0.01
                    nx = x + random.normalvariate(0.0, noise_std + drift_factor)
                    ny = y + random.normalvariate(0.0, noise_std + drift_factor)
                else:
                    nx = x + random.normalvariate(0.0, noise_std)
                    ny = y + random.normalvariate(0.0, noise_std)
                coords.append((max(0.0, min(1.0, nx)), max(0.0, min(1.0, ny))))
            average_error = random.uniform(0.8, 5.0)

        velocities = []
        for i in range(1, len(coords)):
            velocities.append((coords[i][0] - coords[i-1][0], coords[i][1] - coords[i-1][1]))

        jitter_scores = []
        for i in range(1, len(velocities)):
            dv_x = velocities[i][0] - velocities[i-1][0]
            dv_y = velocities[i][1] - velocities[i-1][1]
            jitter_scores.append(math.sqrt(dv_x**2 + dv_y**2))

        track_sample = {
            "track_name": f"track_{idx_t:03d}",
            "region": region,
            "positions": coords,
            "velocities": velocities,
            "jitter_scores": jitter_scores,
            "lifespan": len(coords),
            "survived": survived,
            "has_bundle": has_bundle,
            "average_error": average_error
        }
        track_samples.append(track_sample)

    total_tracks = len(track_samples)
    bundle_ratio = 0.0
    solve_success = False
    solve_error = 99.0

    if total_tracks > 0:
        bundle_count = sum(1 for t in track_samples if t["has_bundle"])
        bundle_ratio = bundle_count / total_tracks
        if bundle_count >= 8 and bundle_ratio >= 0.40:
            solve_success = True
            solve_error = max(0.1, random.normalvariate(0.4 + noise_std * 100, 0.15))
            if tripod:
                solve_error *= 0.8

    settings_dict = {
        "quality_preset": quality,
        "footage_type": "AUTO",
        "robust_mode": robust,
        "tripod_mode": tripod,
        "pattern_size": 11 if quality == "FAST" else (17 if quality == "BALANCED" else 31),
        "search_size": 51 if quality == "FAST" else (71 if quality == "BALANCED" else 121),
        "correlation": 0.55 if quality == "FAST" else (0.70 if quality == "BALANCED" else 0.85),
        "threshold": 0.40 if quality == "FAST" else (0.30 if quality == "BALANCED" else 0.15),
        "motion_model": "LocRot"
    }

    solve_sample = {
        "clip_metadata": meta,
        "settings": settings_dict,
        "tracks": track_samples,
        "solve_success": solve_success,
        "solve_error": solve_error,
        "bundle_count": sum(1 for t in track_samples if t["has_bundle"]),
        "bundle_ratio": bundle_ratio,
        "runtime_seconds": random.uniform(2.0, 15.0)
    }
    return solve_sample

def generate_synthetic_track(idx_t, survived):
    x, y = random.uniform(0.1, 0.9), random.uniform(0.1, 0.9)
    region = classify_region(x, y)
    lifespan = 100 if survived else random.randint(15, 95)
    positions = [(x, y)]
    step_std = 0.002
    for _ in range(1, lifespan):
        dx = random.normalvariate(0.0, step_std)
        dy = random.normalvariate(0.0, step_std)
        if not survived and _ >= lifespan - 5:
            dx += random.uniform(-0.015, 0.015)
            dy += random.uniform(-0.015, 0.015)
        x = max(0.0, min(1.0, x + dx))
        y = max(0.0, min(1.0, y + dy))
        positions.append((x, y))

    velocities = []
    for i in range(1, len(positions)):
        velocities.append((positions[i][0] - positions[i-1][0], positions[i][1] - positions[i-1][1]))

    jitter_scores = []
    for i in range(1, len(velocities)):
        dv_x = velocities[i][0] - velocities[i-1][0]
        dv_y = velocities[i][1] - velocities[i-1][1]
        jitter_scores.append(math.sqrt(dv_x**2 + dv_y**2))

    has_bundle = survived and (random.random() < 0.9)
    average_error = random.uniform(0.15, 0.45) if survived else random.uniform(0.8, 5.0)

    return {
        "track_name": f"t_{idx_t}",
        "region": region,
        "positions": positions,
        "velocities": velocities,
        "jitter_scores": jitter_scores,
        "lifespan": lifespan,
        "survived": survived,
        "has_bundle": has_bundle,
        "average_error": average_error
    }

def simulate_solve_attempts(clips_dir=CLIPS_DIR, out_dir=RAW_DIR, force_reprocess=False):
    # Setup local write cache on Google Drive to speed up writes
    write_raw_dir = out_dir
    is_colab_drive = in_colab and USE_GOOGLE_DRIVE and out_dir == RAW_DIR
    if is_colab_drive:
        write_raw_dir = "/content/autosolve_raw_cache"
        os.makedirs(write_raw_dir, exist_ok=True)
        if os.path.exists(out_dir):
            print(f"Syncing existing metadata files from Drive to local cache: {write_raw_dir}...")
            import shutil
            for f in os.listdir(out_dir):
                if f.endswith('.json'):
                    try:
                        shutil.copy2(os.path.join(out_dir, f), os.path.join(write_raw_dir, f))
                    except Exception:
                        pass
    else:
        os.makedirs(out_dir, exist_ok=True)
    supported = {'.mp4', '.mov', '.avi', '.mkv', '.ogg', '.webm'}
    files = [os.path.join(clips_dir, f) for f in os.listdir(clips_dir) if os.path.splitext(f)[1].lower() in supported] if os.path.exists(clips_dir) else []

    if not files:
        print("No clips found in clips directory. Generating synthetic trajectories to allow end-to-end training...")
        for i in range(5):
            clip_name = f"dummy_clip_{i}"
            with open(os.path.join(write_raw_dir, f"{clip_name}_video_meta.json"), 'w') as fh:
                json.dump({"clip_name": clip_name, "width": 1920, "height": 1080, "fps": 30.0, "frame_count": 250, "mean_motion": 0.6, "zoom_divergence": 0.0, "distortion_factor": 0.0, "noise_ratio": 0.003}, fh, indent=4)

            dummy_tracks = []
            for j in range(40):
                survived = (j % 2 == 0)
                dummy_tracks.append(generate_synthetic_track(j, survived))

            solve = {
                "clip_metadata": {"clip_name": clip_name, "width": 1920, "height": 1080, "fps": 30.0, "frame_count": 250},
                "settings": {"quality_preset": "BALANCED", "footage_type": "AUTO", "robust_mode": False, "tripod_mode": False, "pattern_size": 17, "search_size": 71, "correlation": 0.7, "threshold": 0.3, "motion_model": "LocRot"},
                "tracks": dummy_tracks,
                "solve_success": True, "solve_error": 0.3, "bundle_count": 20, "bundle_ratio": 0.5, "runtime_seconds": 5.0
            }
            with open(os.path.join(write_raw_dir, f"{clip_name}_balanced_standard.json"), 'w') as fh:
                json.dump(solve, fh, indent=4)
        return

    print(f"Processing {len(files)} video clips...")
    for fp in files:
        clip_name = os.path.splitext(os.path.basename(fp))[0]
        expected_outputs = [
            os.path.join(write_raw_dir, f"{clip_name}_{suffix}.json")
            for suffix in ["balanced_standard", "fast_standard", "quality_standard", "balanced_robust", "balanced_tripod"]
        ]
        expected_outputs.append(os.path.join(write_raw_dir, f"{clip_name}_video_meta.json"))

        if not force_reprocess and all(os.path.exists(p) for p in expected_outputs):
            print(f"  Clip '{clip_name}' already processed. Skipping feature extraction.")
            continue
        try:
            print(f"  Extracting video features from: {os.path.basename(fp)}")
            v_feats = extract_features_from_video(fp)

            # YOLOv8 semantic masking
            semantic_fp = os.path.join(write_raw_dir, f"{clip_name}_semantic_meta.json")
            run_yolo_semantic_masking(fp, semantic_fp)

            v_feats["dynamic_area_ratio"] = 0.0
            if os.path.exists(semantic_fp):
                ratio = compute_dynamic_pixel_ratio(semantic_fp)
                if ratio is not None:
                    v_feats["dynamic_area_ratio"] = ratio

            # Offload YOLOv8 from GPU VRAM by clearing cache
            global _yolo_model_cache
            _yolo_model_cache = None
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            with open(os.path.join(write_raw_dir, f"{clip_name}_video_meta.json"), 'w', encoding='utf-8') as fh:
                json.dump(v_feats, fh, indent=4)

            # Trajectory tracking using CoTracker (if available) or OpenCV (fallback)
            has_cotracker = False
            try:
                import cotracker
                has_cotracker = True
            except ImportError:
                pass

            if has_cotracker:
                print(f"  Running CoTracker v3 tracking: {os.path.basename(fp)}")
                trajectories, meta = run_cotracker_tracking(fp)
            else:
                print("  [Info] CoTracker v3 is not installed or import failed.")
                print("         To enable high-accuracy CoTracker tracking on Colab, uncomment and run the '!pip install git+...' line in the dependencies cell and restart the runtime.")
                print(f"  Running OpenCV optical flow tracking: {os.path.basename(fp)}")
                trajectories, meta = run_opencv_tracking(fp)

            variations = [
                ("BALANCED", False, False, "balanced_standard"),
                ("FAST", False, False, "fast_standard"),
                ("QUALITY", False, False, "quality_standard"),
                ("BALANCED", True, False, "balanced_robust"),
                ("BALANCED", False, True, "balanced_tripod")
            ]
            for q, r, t, suffix in variations:
                solve = simulate_variation(trajectories, meta, q, r, t)
                with open(os.path.join(write_raw_dir, f"{clip_name}_{suffix}.json"), 'w') as fh:
                    json.dump(solve, fh, indent=4)
        except Exception as e:
            print(f"  Failed {clip_name}: {e}")
        finally:
            import gc
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    if is_colab_drive:
            print(f"Syncing local cache back to Google Drive raw directory: {out_dir}...")
            import shutil
            os.makedirs(out_dir, exist_ok=True)
            for f in os.listdir(write_raw_dir):
                if f.endswith('.json'):
                    try:
                        shutil.copy2(os.path.join(write_raw_dir, f), os.path.join(out_dir, f))
                    except Exception:
                        pass
            print("Sync complete.")

def prepare_dataset(data_dir=RAW_DIR, output_json=None, seed=42):
    if output_json is None:
        output_json = os.path.join(PROCESSED_DIR, "settings_dataset.json")
    video_meta_lookup = {}
    real_samples = []
    
    if os.path.exists(data_dir):
        for f in os.listdir(data_dir):
            fp = os.path.join(data_dir, f)
            if f.endswith('_video_meta.json'):
                try:
                    with open(fp) as fh:
                        m = json.load(fh)
                        c_name = m.get("clip_name")
                        if c_name:
                            # Overwrite noise_ratio with dynamic_pixel_ratio if semantic features exist
                            semantic_fp = os.path.join(data_dir, f"{c_name}_semantic_meta.json")
                            if os.path.exists(semantic_fp):
                                try:
                                    with open(semantic_fp, 'r', encoding='utf-8') as sfh:
                                        sem_d = json.load(sfh)
                                    frames = sem_d.get("frames", [])
                                    if frames:
                                        total_ratio = 0.0
                                        for frame in frames:
                                            detections = frame.get("detections", [])
                                            frame_dynamic_area = 0.0
                                            for det in detections:
                                                if det.get("is_dynamic", False):
                                                    box = det.get("box", [0.0, 0.0, 0.0, 0.0])
                                                    w = max(0.0, box[2] - box[0])
                                                    h = max(0.0, box[3] - box[1])
                                                    frame_dynamic_area += w * h
                                            total_ratio += min(1.0, frame_dynamic_area)
                                        m["noise_ratio"] = round(total_ratio / len(frames), 4)
                                except Exception as se:
                                    print(f"Warning: failed to compute dynamic pixel ratio for {c_name}: {se}")
                        video_meta_lookup[c_name] = m
                except Exception as e:
                    print(f"Warning: failed to load video meta {f}: {e}")
            elif is_solve_json(f):
                try:
                    with open(fp) as fh:
                        s = json.load(fh)
                        real_samples.append(s)
                except Exception as e:
                    print(f"Warning: failed to load solve sample {fp}: {e}")

    by_clip = {}
    for s in real_samples:
        if "clip_metadata" in s:
            by_clip.setdefault(s["clip_metadata"]["clip_name"], []).append(s)

    clip_names = list(by_clip.keys())
    random.seed(seed)
    train_raw, val_raw = [], []

    if len(clip_names) > 1:
        random.shuffle(clip_names)
        split_idx = max(1, int(len(clip_names) * 0.8))
        train_clips = set(clip_names[:split_idx])
        for c, samples in by_clip.items():
            if c in train_clips:
                train_raw.extend(samples)
            else:
                val_raw.extend(samples)
    else:
        if clip_names:
            all_samples = by_clip[clip_names[0]]
            random.shuffle(all_samples)
            split_idx = max(1, int(len(all_samples) * 0.8))
            train_raw = all_samples[:split_idx]
            val_raw = all_samples[split_idx:]

    if not train_raw:
        print("No samples available. Generating synthetic settings dataset...")
        train_X = [np.random.randn(28).tolist() for _ in range(100)]
        train_y = [random.random() for _ in range(100)]
        val_X = [np.random.randn(28).tolist() for _ in range(25)]
        val_y = [random.random() for _ in range(25)]
    else:
        train_X = [extract_sample_features(s, video_meta_lookup.get(s["clip_metadata"]["clip_name"])) for s in train_raw]
        train_y = [calculate_reward(s) for s in train_raw]
        val_X = [extract_sample_features(s, video_meta_lookup.get(s["clip_metadata"]["clip_name"])) for s in val_raw]
        val_y = [calculate_reward(s) for s in val_raw]

    num_features = len(train_X[0])
    means, stds = [0.0]*num_features, [1.0]*num_features
    for j in range(num_features):
        if j in (4, 5) or (10 <= j < 24):
            means[j] = 0.0
            stds[j] = 1.0
            continue
        col = [train_X[i][j] for i in range(len(train_X))]
        means[j] = sum(col) / len(col)
        variance = sum((x - means[j])**2 for x in col) / len(col)
        stds[j] = math.sqrt(variance) if variance > 1e-8 else 1.0

    def normalize(X):
        return [[(val - m)/s for val, m, s in zip(row, means, stds)] for row in X]

    dataset = {
        "train": {"X": normalize(train_X), "y": train_y},
        "val": {"X": normalize(val_X), "y": val_y},
        "input_mean": means, "input_std": stds
    }
    os.makedirs(os.path.dirname(output_json), exist_ok=True)
    with open(output_json, 'w') as fh:
        json.dump(dataset, fh, indent=4)
    print(f"Ingestion completed. Dataset saved to {output_json}")


In [ ]:
# Execute Ingestion Pipeline
# Set force_reprocess=True if you want to overwrite and reprocess already completed clips
simulate_solve_attempts(force_reprocess=False)
prepare_dataset()


In [ ]:
# Train Expected Reward Optimizer MLP
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

class SettingsMLP(nn.Module):
    """Expected reward MLP mapping 29 features to 1 expected reward score."""
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(29, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.network(x)

def train_settings_optimizer(epochs=50):
    set_seed(42)
    dataset_path = os.path.join(PROCESSED_DIR, "settings_dataset.json")
    with open(dataset_path) as f:
        dataset = json.load(f)
        
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Training SettingsMLP on device: {device}")
    tx = torch.tensor(dataset["train"]["X"], dtype=torch.float32).to(device)
    ty = torch.tensor(dataset["train"]["y"], dtype=torch.float32).unsqueeze(1).to(device)
    vx = torch.tensor(dataset["val"]["X"], dtype=torch.float32).to(device)
    vy = torch.tensor(dataset["val"]["y"], dtype=torch.float32).unsqueeze(1).to(device)

    loader = DataLoader(TensorDataset(tx, ty), batch_size=16, shuffle=True)
    model = SettingsMLP().to(device)
    criterion = nn.SmoothL1Loss()
    optimizer = optim.AdamW(model.parameters(), lr=0.005, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    best_val_loss = float('inf')
    best_weights = None

    train_losses = []
    val_losses = []

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for bx, by in loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            # Data Augmentation: add low-magnitude Gaussian noise to continuous features
            if model.training:
                noise = torch.randn_like(bx) * 0.01
                for col_idx in range(bx.shape[1]):
                    if col_idx in (4, 5) or (10 <= col_idx < 24):
                        noise[:, col_idx] = 0.0
                bx_augmented = bx + noise
            else:
                bx_augmented = bx
            loss = criterion(model(bx_augmented), by)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item() * bx.size(0)
        train_loss /= len(tx)
        train_losses.append(train_loss)
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(vx), vy).item()
        val_losses.append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights = {k: v.cpu().numpy().tolist() for k, v in model.state_dict().items()}

        if (epoch+1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:02d}/{epochs} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}")

    meta = {"weights": best_weights, "input_mean": dataset["input_mean"], "input_std": dataset["input_std"], "best_val_loss": best_val_loss}
    save_dir = os.path.join(RUNS_DIR, "settings_optimizer")
    os.makedirs(save_dir, exist_ok=True)
    with open(os.path.join(save_dir, "model_meta_weights.json"), 'w') as fh:
        json.dump(meta, fh, indent=4)
    print("Settings Optimizer weights saved successfully.")

    # Visualise training curve
    plt.figure(figsize=(8, 4))
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses, label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Settings Expected-Reward Model Training")
    plt.legend()
    plt.grid(True)
    plt.show()

train_settings_optimizer()


In [ ]:
# Train Track Quality Predictor MLP

class TrackMLP(nn.Module):
    """3-layer MLP for track survival prediction."""
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(15, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.network(x)

def train_track_predictor(epochs=100):
    set_seed(42)
    data_dir = RAW_DIR
    json_files = [os.path.join(data_dir, f) for f in os.listdir(data_dir) if is_solve_json(f)] if os.path.exists(data_dir) else []

    X_samples, y_samples = [], []
    clip_ids = []
    FOOTAGE_MAP = {f: i for i, f in enumerate(FOOTAGE_TYPES)}
    REGIONS_LIST = ['top-left', 'top-center', 'top-right', 'mid-left', 'center', 'mid-right', 'bottom-left', 'bottom-center', 'bottom-right']

    for fp in json_files:
        try:
            clip_name = os.path.splitext(os.path.basename(fp))[0]
            with open(fp) as fh:
                d = json.load(fh)
            footage_idx = FOOTAGE_MAP.get(d["settings"].get("footage_type", "AUTO"), 0)
            robust = d["settings"].get("robust_mode", False)
            tracks = d["tracks"]
            t_coords = {t["track_name"]: t["positions"] for t in tracks}
            for t in tracks:
                coords = t["positions"]
                survived = t["survived"] and t["has_bundle"]
                r_idx = REGIONS_LIST.index(t["region"]) if t["region"] in REGIONS_LIST else 4
                if len(coords) < 6:
                    continue
                for f_idx_check in range(5, len(coords), 10):
                    sub = coords[:f_idx_check+1]
                    neighbors = [tc[:f_idx_check+1] for name, tc in t_coords.items() if name != t["track_name"] and len(tc) > f_idx_check]
                    feats = np.zeros(15, dtype=np.float32)
                    inspect = sub[-6:]
                    v_x = [inspect[i][0] - inspect[i-1][0] for i in range(1, len(inspect)) ]
                    v_y = [inspect[i][1] - inspect[i-1][1] for i in range(1, len(inspect)) ]
                    feats[0], feats[1] = np.mean(v_x), np.mean(v_y)
                    feats[2] = np.std(v_x) if len(v_x) > 1 else 0.0
                    feats[3] = np.std(v_y) if len(v_y) > 1 else 0.0
                    acc_x = [v_x[i] - v_x[i-1] for i in range(1, len(v_x))] if len(v_x) > 1 else [0.0]
                    acc_y = [v_y[i] - v_y[i-1] for i in range(1, len(v_y))] if len(v_y) > 1 else [0.0]
                    feats[4], feats[5] = np.mean(acc_x), np.mean(acc_y)
                    feats[6] = sum(1 for i in range(1, len(v_x)) if (v_x[i] > 0) != (v_x[i-1] > 0))
                    feats[7] = sum(1 for i in range(1, len(v_y)) if (v_y[i] > 0) != (v_y[i-1] > 0))
                    feats[8] = len(sub)  # FIXED: Use length so far instead of total length
                    min_dist = 999.0
                    n_vels_x, n_vels_y = [], []
                    for n_coords in neighbors:
                        if len(n_coords) >= 1:
                            # FIXED: Compute distance using current position sub[-1] instead of future final position coords[-1]
                            dist = ((sub[-1][0] - n_coords[-1][0])**2 + (sub[-1][1] - n_coords[-1][1])**2)**0.5
                            if dist < min_dist:
                                min_dist = dist
                            if len(n_coords) >= 2:
                                n_vels_x.append(n_coords[-1][0] - n_coords[-2][0])
                                n_vels_y.append(n_coords[-1][1] - n_coords[-2][1])
                    feats[9] = min_dist if min_dist < 998.0 else 1.0
                    if n_vels_x:
                        feats[10] = v_x[-1] - np.mean(n_vels_x)
                        feats[11] = v_y[-1] - np.mean(n_vels_y)
                    feats[12], feats[13] = float(r_idx), float(footage_idx)
                    feats[14] = 1.0 if robust else 0.0

                    X_samples.append(feats)
                    y_samples.append(1.0 if survived else 0.0)
                    clip_ids.append(clip_name)
        except Exception as e:
            print(f"Skipped {fp}: {e}")

    if not X_samples:
        print("No track samples found. Injecting synthetic vectors for training...")
        for i in range(5):
            clip_name = f"dummy_clip_{i}"
            for j in range(40):
                survived = (j % 2 == 0)
                t = generate_synthetic_track(j, survived)
                coords = t["positions"]
                r_idx = REGIONS_LIST.index(t["region"]) if t["region"] in REGIONS_LIST else 4
                if len(coords) < 6:
                    continue
                for f_idx_check in range(5, len(coords), 10):
                    sub = coords[:f_idx_check+1]
                    feats = np.zeros(15, dtype=np.float32)
                    inspect = sub[-6:]
                    v_x = [inspect[i][0] - inspect[i-1][0] for i in range(1, len(inspect)) ]
                    v_y = [inspect[i][1] - inspect[i-1][1] for i in range(1, len(inspect)) ]
                    feats[0], feats[1] = np.mean(v_x), np.mean(v_y)
                    feats[2] = np.std(v_x) if len(v_x) > 1 else 0.0
                    feats[3] = np.std(v_y) if len(v_y) > 1 else 0.0
                    acc_x = [v_x[i] - v_x[i-1] for i in range(1, len(v_x))] if len(v_x) > 1 else [0.0]
                    acc_y = [v_y[i] - v_y[i-1] for i in range(1, len(v_y))] if len(v_y) > 1 else [0.0]
                    feats[4], feats[5] = np.mean(acc_x), np.mean(acc_y)
                    feats[6] = sum(1 for i in range(1, len(v_x)) if (v_x[i] > 0) != (v_x[i-1] > 0))
                    feats[7] = sum(1 for i in range(1, len(v_y)) if (v_y[i] > 0) != (v_y[i-1] > 0))
                    feats[8] = len(sub)  # FIXED: Use length so far
                    feats[9] = 0.1
                    feats[10] = 0.0
                    feats[11] = 0.0
                    feats[12], feats[13] = float(r_idx), 0.0
                    feats[14] = 0.0
                    X_samples.append(feats)
                    y_samples.append(1.0 if survived else 0.0)
                    clip_ids.append(clip_name)

    X_arr = np.array(X_samples, dtype=np.float32)
    y_arr = np.array(y_samples, dtype=np.float32)

    pos_ratio = sum(y_samples) / len(y_samples) if y_samples else 0.5
    print(f"Class balance: {pos_ratio:.1%} survived")
    pos_weight = torch.tensor([(1.0 - pos_ratio) / max(1e-5, pos_ratio)], dtype=torch.float32)

    unique_clips = list(set(clip_ids))
    random.seed(42)
    random.shuffle(unique_clips)
    split_idx = max(1, int(len(unique_clips) * 0.8))
    train_clips = set(unique_clips[:split_idx])

    train_indices = [i for i, c in enumerate(clip_ids) if c in train_clips]
    val_indices = [i for i, c in enumerate(clip_ids) if c not in train_clips]

    X_train_raw = X_arr[train_indices]
    y_train = y_arr[train_indices]
    X_val_raw = X_arr[val_indices]
    y_val = y_arr[val_indices]

    mean = np.mean(X_train_raw, axis=0) if len(X_train_raw) > 0 else np.zeros(15)
    std = np.std(X_train_raw, axis=0) if len(X_train_raw) > 0 else np.ones(15)
    std[std < 1e-6] = 1.0

    X_train_norm = (X_train_raw - mean) / std if len(X_train_raw) > 0 else X_train_raw
    X_val_norm = (X_val_raw - mean) / std if len(X_val_raw) > 0 else X_val_raw

    tx, ty = torch.tensor(X_train_norm), torch.tensor(y_train).unsqueeze(1)
    vx, vy = torch.tensor(X_val_norm), torch.tensor(y_val).unsqueeze(1)

    loader = DataLoader(TensorDataset(tx, ty), batch_size=32, shuffle=True)
    model = TrackMLP()
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    best_val_loss = float('inf')
    best_weights = None
    best_train_thresh = 0.5

    train_losses = []
    val_losses = []
    val_accuracies = []

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for bx, by in loader:
            optimizer.zero_grad()
            if model.training:
                noise = torch.randn_like(bx) * 0.01
                for col_idx in range(bx.shape[1]):
                    if col_idx in (12, 13, 14):
                        noise[:, col_idx] = 0.0
                bx_augmented = bx + noise
            else:
                bx_augmented = bx
            loss = criterion(model(bx_augmented), by)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()*bx.size(0)
        train_loss /= len(tx)
        train_losses.append(train_loss)
        scheduler.step()

        model.eval()
        with torch.no_grad():
            logits = model(vx)
            val_loss = criterion(logits, vy).item()
            probs = torch.sigmoid(logits)
            
            train_logits = model(tx)
            train_probs = torch.sigmoid(train_logits).numpy().flatten()
            y_train_np = ty.numpy().flatten()
            
            best_thresh = 0.5
            best_tr_acc = 0.0
            for t_val in np.linspace(0.1, 0.9, 81):
                tr_acc = np.mean((train_probs > t_val) == y_train_np)
                if tr_acc > best_tr_acc:
                    best_tr_acc = tr_acc
                    best_thresh = t_val
            
            if len(vy) > 0:
                y_val_np = vy.numpy().flatten()
                probs_np = probs.numpy().flatten()
                
                acc_05 = np.mean((probs_np > 0.5) == y_val_np)
                preds_opt = (probs_np > best_thresh).astype(np.float32)
                acc_opt = np.mean(preds_opt == y_val_np)
                
                tp = np.sum((y_val_np == 1.0) & (preds_opt == 1.0))
                tn = np.sum((y_val_np == 0.0) & (preds_opt == 0.0))
                fp = np.sum((y_val_np == 0.0) & (preds_opt == 1.0))
                fn = np.sum((y_val_np == 1.0) & (preds_opt == 0.0))
                
                sens = tp / max(1, tp + fn)
                spec = tn / max(1, tn + fp)
                bal_acc = (sens + spec) / 2.0
                f1_score = 2.0 * tp / max(1e-8, 2.0 * tp + fp + fn)
            else:
                acc_05 = 1.0
                acc_opt = 1.0
                bal_acc = 1.0
                f1_score = 1.0

        val_losses.append(val_loss)
        val_accuracies.append(acc_opt)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights = {k: v.cpu().numpy().tolist() for k, v in model.state_dict().items()}
            best_train_thresh = best_thresh
            
        if (epoch+1)%20 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:03d}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
                  f"Val Acc (0.5): {acc_05:.1%} | Val Acc (Opt @ {best_thresh:.2f}): {acc_opt:.1%} | "
                  f"Val BalAcc: {bal_acc:.1%} | Val F1: {f1_score:.1%}")

    import math
    logit_shift = math.log(best_train_thresh / max(1e-5, 1.0 - best_train_thresh))
    if best_weights and "network.8.bias" in best_weights:
        best_weights["network.8.bias"][0] -= logit_shift
        print(f"\nShifted final layer bias by {logit_shift:.4f} to align optimal threshold ({best_train_thresh:.2f}) with 0.5 for exporting.")

    meta = {"weights": best_weights, "input_mean": mean.tolist(), "input_std": std.tolist()}
    save_dir = os.path.join(RUNS_DIR, "track_predictor")
    os.makedirs(save_dir, exist_ok=True)
    with open(os.path.join(save_dir, "model_meta_weights.json"), 'w') as fh:
        json.dump(meta, fh, indent=4)
    print("Track Predictor saved successfully.")

    fig, ax1 = plt.subplots(figsize=(8, 4))
    ax1.plot(train_losses, label="Train Loss", color="blue")
    ax1.plot(val_losses, label="Val Loss", color="red")
    ax1.set_ylabel("Loss")
    ax1.legend(loc="upper left")

    ax2 = ax1.twinx()
    ax2.plot(val_accuracies, label="Val Acc (Opt)", color="green", linestyle="--")
    ax2.set_ylabel("Accuracy")
    ax2.legend(loc="upper right")
    plt.title("Track Predictor Training Curves (Optimized Threshold)")
    plt.show()

train_track_predictor()

In [ ]:
# Train Patch Rigidity Classifier CNN

class PatchRigidityCNN(nn.Module):
    """Lightweight 2-layer CNN for 32x32 patch rigidity classification."""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # 32x32 -> 16x16
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  # 16x16 -> 8x8
        )
        self.classifier = nn.Sequential(
            nn.Linear(32 * 8 * 8, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

def generate_synthetic_patch_data(num_samples=200):
    print("Generating synthetic patch rigidity training data...")
    X = []
    y = []
    for i in range(num_samples):
        label = 1.0 if random.random() < 0.5 else 0.0
        patch = np.zeros((32, 32), dtype=np.float32)
        if label == 1.0:
            x_line = random.randint(5, 25)
            y_line = random.randint(5, 25)
            patch[x_line-2:x_line+2, :] = 1.0
            patch[:, y_line-2:y_line+2] = 1.0
            patch += np.random.normal(0.0, 0.05, (32, 32))
        else:
            if random.random() < 0.5:
                patch += random.uniform(0.1, 0.8)
                patch += np.random.normal(0.0, 0.005, (32, 32))
            else:
                patch += np.random.normal(0.5, 0.25, (32, 32))
        patch = np.clip(patch, 0.0, 1.0)
        X.append(patch)
        y.append(label)
    X = np.expand_dims(np.array(X, dtype=np.float32), 1)
    y = np.array(y, dtype=np.float32)
    clip_ids = [f"synth_clip_{i // 40}" for i in range(num_samples)]
    return X, y, clip_ids

def extract_real_patch_data(clips_dir, data_dir):
    solve_files = [f for f in os.listdir(data_dir) if is_solve_json(f)]
    if not solve_files:
        return None, None, None
        
    X_list = []
    y_list = []
    clip_ids = []
    
    for file_name in solve_files:
        solve_path = os.path.join(data_dir, file_name)
        try:
            with open(solve_path, 'r', encoding='utf-8') as f:
                solve_data = json.load(f)
                
            clip_name = solve_data["clip_metadata"]["clip_name"]
            # Try to locate video clip
            video_path = os.path.join(clips_dir, clip_name)
            if not os.path.exists(video_path):
                base_name = os.path.splitext(clip_name)[0]
                matches = [f for f in os.listdir(clips_dir) if f.startswith(base_name)]
                if matches:
                    video_path = os.path.join(clips_dir, matches[0])
                else:
                    continue
                    
            print(f"  Segmenting motion via RANSAC on: {os.path.basename(video_path)}")
            cap = cv2.VideoCapture(video_path)
            if not cap.isOpened(): continue
            try:
                width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
                height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
                
                tracks = solve_data["tracks"]
                track_points = {t["track_name"]: {idx: pos for idx, pos in enumerate(t["positions"])} for t in tracks}
                num_frames = max(len(t["positions"]) for t in tracks)
                track_outlier_votes = {t["track_name"]: {"outlier": 0, "total": 0} for t in tracks}
                
                for f_idx in range(num_frames - 1):
                    src_pts, dst_pts, names = [], [], []
                    for name, pos_map in track_points.items():
                        if f_idx in pos_map and (f_idx + 1) in pos_map:
                            x0, y0 = pos_map[f_idx]
                            x1, y1 = pos_map[f_idx + 1]
                            src_pts.append([x0 * width, y0 * height])
                            dst_pts.append([x1 * width, y1 * height])
                            names.append(name)
                            
                    if len(src_pts) >= 8:
                        src_pts = np.array(src_pts, dtype=np.float32)
                        dst_pts = np.array(dst_pts, dtype=np.float32)
                        H, inliers = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)
                        if inliers is not None:
                            for i, name in enumerate(names):
                                track_outlier_votes[name]["total"] += 1
                                if inliers[i][0] == 0:
                                    track_outlier_votes[name]["outlier"] += 1
                                    
                track_rigidity = {}
                for name, votes in track_outlier_votes.items():
                    if votes["total"] > 0:
                        outlier_ratio = votes["outlier"] / votes["total"]
                        track_rigidity[name] = 0.0 if outlier_ratio > 0.35 else 1.0
                    else:
                        track_rigidity[name] = 1.0
                        
                frame_idx = 0
                while True:
                    ret, frame = cap.read()
                    if not ret: break
                    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                    for name, pos_map in track_points.items():
                        if frame_idx in pos_map and frame_idx % 10 == 0:
                            x, y = pos_map[frame_idx]
                            px, py = int(x * width), int(y * height)
                            y0, y1 = py - 16, py + 16
                            x0, x1 = px - 16, px + 16
                            if y0 >= 0 and y1 < height and x0 >= 0 and x1 < width:
                                patch = gray[y0:y1, x0:x1].astype(np.float32) / 255.0
                                X_list.append(patch)
                                y_list.append(track_rigidity[name])
                                clip_ids.append(clip_name)
                    frame_idx += 1
            finally:
                cap.release()
        except Exception as e:
            print(f"  Error processing video patch generation: {e}")
            
    if not X_list:
        return None, None, None
    X = np.expand_dims(np.array(X_list, dtype=np.float32), 1)
    y = np.array(y_list, dtype=np.float32)
    return X, y, clip_ids

def train_patch_rigidity(epochs=30, batch_size=16, learning_rate=0.001):
    X, y, clip_ids = None, None, None
    if os.path.exists(CLIPS_DIR) and os.path.exists(RAW_DIR):
        video_extensions = {'.mp4', '.mov', '.avi', '.mkv', '.ogg', '.webm'}
        video_files = [f for f in os.listdir(CLIPS_DIR) if os.path.splitext(f)[1].lower() in video_extensions]
        if video_files:
            X, y, clip_ids = extract_real_patch_data(CLIPS_DIR, RAW_DIR)
            
    if X is None:
        X, y, clip_ids = generate_synthetic_patch_data(num_samples=400)
        
    print(f"Training patch rigidity dataset: {len(X)} patches. Rigid ratio: {np.mean(y):.1%}")
    
    unique_clips = list(set(clip_ids))
    random.seed(42)
    random.shuffle(unique_clips)
    split_idx = max(1, int(len(unique_clips) * 0.8))
    train_clips = set(unique_clips[:split_idx])
    
    train_idx = [i for i, c in enumerate(clip_ids) if c in train_clips]
    val_idx = [i for i, c in enumerate(clip_ids) if c not in train_clips]
    
    train_dataset = TensorDataset(torch.tensor(X[train_idx]), torch.tensor(y[train_idx]).unsqueeze(1))
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    model = PatchRigidityCNN()
    criterion = nn.BCELoss()
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    
    best_loss = 9999.0
    best_weights = None
    
    train_losses = []
    val_losses = []
    
    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item() * batch_x.size(0)
            
        train_loss /= len(train_idx)
        train_losses.append(train_loss)
        scheduler.step()
        
        model.eval()
        with torch.no_grad():
            val_X_t = torch.tensor(X[val_idx])
            val_y_t = torch.tensor(y[val_idx]).unsqueeze(1)
            val_preds = model(val_X_t)
            val_loss = criterion(val_preds, val_y_t).item()
            acc = np.mean((val_preds.numpy() > 0.5) == y[val_idx].reshape(-1, 1)) if len(val_idx) > 0 else 1.0
        val_losses.append(val_loss)
            
        if epoch % 10 == 0 or epoch == 1:
            print(f"Epoch {epoch:2d}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {acc:.1%}")
            
        if val_loss < best_loss:
            best_loss = val_loss
            best_weights = {k: v.clone() for k, v in model.state_dict().items()}
            
    if best_weights is not None:
        model.load_state_dict(best_weights)
        print(f"Best Validation Loss: {best_loss:.4f}")
            
    # Export to ONNX
    onnx_path = os.path.join(ONNX_OUT_DIR, "patch_rigidity.onnx")
    dummy_input = torch.zeros(1, 1, 32, 32, dtype=torch.float32)
    torch.onnx.export(
        model, dummy_input, onnx_path,
        input_names=["patch_pixels"], output_names=["rigidity_score"], opset_version=17,
        dynamic_axes={"patch_pixels": {0: "batch"}, "rigidity_score": {0: "batch"}}
    )
    meta = {
        "input_size": [32, 32],
        "channels": 1,
        "normalization": "div_255",
        "description": "AutoSolve Patch Rigidity CNN. Inputs 32x32 grayscale image patches scaled to [0,1]. Outputs static/rigid rigidity probability in [0,1]."
    }
    with open(os.path.join(ONNX_OUT_DIR, "patch_rigidity_meta.json"), 'w') as fh:
        json.dump(meta, fh, indent=4)
        
    print(f"ONNX Model saved to: {onnx_path}")
    
    # Plot curves
    plt.figure(figsize=(8, 4))
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses, label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Patch Rigidity CNN Training")
    plt.legend()
    plt.grid(True)
    plt.show()

train_patch_rigidity()


In [ ]:
# Compile Region Weights Heatmap

def compile_region_weights(data_dir=RAW_DIR, output_json=None):
    if output_json is None:
        output_json = os.path.join(RUNS_DIR, "region_weights.json")
    regions = ['top-left', 'top-center', 'top-right', 'mid-left', 'center', 'mid-right', 'bottom-left', 'bottom-center', 'bottom-right']
    prebaked = {
        "AUTO": {"top-left": 0.535, "top-center": 0.674, "top-right": 0.490, "mid-left": 0.768, "center": 1.000, "mid-right": 0.751, "bottom-left": 0.776, "bottom-center": 0.978, "bottom-right": 0.800},
        "INDOOR": {"top-left": 0.850, "top-center": 0.800, "top-right": 0.850, "mid-left": 0.950, "center": 1.000, "mid-right": 0.950, "bottom-left": 0.900, "bottom-center": 0.900, "bottom-right": 0.900},
        "OUTDOOR": {"top-left": 0.400, "top-center": 0.150, "top-right": 0.400, "mid-left": 0.850, "center": 0.950, "mid-right": 0.850, "bottom-left": 1.000, "bottom-center": 1.000, "bottom-right": 1.000},
        "DRONE": {"top-left": 0.100, "top-center": 0.050, "top-right": 0.100, "mid-left": 0.700, "center": 0.900, "mid-right": 0.700, "bottom-left": 1.000, "bottom-center": 1.000, "bottom-right": 1.000},
        "HANDHELD": {"top-left": 0.500, "top-center": 0.600, "top-right": 0.500, "mid-left": 0.800, "center": 1.000, "mid-right": 0.800, "bottom-left": 0.600, "bottom-center": 0.800, "bottom-right": 0.600},
        "GIMBAL": {"top-left": 0.900, "top-center": 0.900, "top-right": 0.900, "mid-left": 0.950, "center": 1.000, "mid-right": 0.950, "bottom-left": 0.950, "bottom-center": 0.950, "bottom-right": 0.950},
        "ACTION": {"top-left": 0.300, "top-center": 0.500, "top-right": 0.300, "mid-left": 0.700, "center": 1.000, "mid-right": 0.700, "bottom-left": 0.400, "bottom-center": 0.700, "bottom-right": 0.400},
        "VFX": {"top-left": 0.800, "top-center": 0.850, "top-right": 0.800, "mid-left": 0.950, "center": 1.000, "mid-right": 0.950, "bottom-left": 0.900, "bottom-center": 0.950, "bottom-right": 0.900},
        "SCREEN": {"top-left": 0.200, "top-center": 0.300, "top-right": 0.200, "mid-left": 0.600, "center": 1.000, "mid-right": 0.600, "bottom-left": 0.200, "bottom-center": 0.400, "bottom-right": 0.200},
        "CINEMATIC": {"top-left": 0.400, "top-center": 0.600, "top-right": 0.400, "mid-left": 0.850, "center": 1.000, "mid-right": 0.850, "bottom-left": 0.500, "bottom-center": 0.800, "bottom-right": 0.500}
    }
    json_files = [os.path.join(data_dir, f) for f in os.listdir(data_dir) if is_solve_json(f)] if os.path.exists(data_dir) else []
    weights = {}
    if json_files:
        stats = {}
        for fp in json_files:
            try:
                with open(fp) as fh: d = json.load(fh)
                f_type = d["settings"].get("footage_type", "AUTO")
                stats.setdefault(f_type, {})
                for t in d["tracks"]:
                    region = t["region"]
                    r_stats = stats[f_type].setdefault(region, {"detected": 0, "survived": 0})
                    r_stats["detected"] += 1
                    if t["survived"] and t["has_bundle"]: r_stats["survived"] += 1
            except: pass

        for f_type in FOOTAGE_TYPES:
            weights[f_type] = {}
            f_stats = stats.get(f_type, {})
            fallback_weights = prebaked.get(f_type, prebaked["AUTO"])
            for region, r_weights in fallback_weights.items():
                if region in f_stats and f_stats[region]["detected"] > 10:
                    weights[f_type][region] = f_stats[region]["survived"] / f_stats[region]["detected"]
                else:
                    weights[f_type][region] = r_weights
            max_w = max(weights[f_type].values())
            if max_w > 0:
                for r in weights[f_type]: weights[f_type][r] = round(weights[f_type][r]/max_w, 3)
    else:
        weights = prebaked

    with open(output_json, 'w') as fh:
        json.dump(weights, fh, indent=4)
    print(f"Empirical weights generated under {output_json}")

compile_region_weights()


In [ ]:
# Presets Grid Search and Model Exporters

def get_resolution_metadata(res_class: str) -> Tuple[float, float, float, float]:
    meta_map = {
        'HD_24fps': (1920.0, 1080.0, 24.0, 250.0),
        'HD_30fps': (1920.0, 1080.0, 30.0, 250.0),
        'HD_60fps': (1920.0, 1080.0, 60.0, 250.0),
        '4K_24fps': (3840.0, 2160.0, 24.0, 250.0),
        '4K_30fps': (3840.0, 2160.0, 30.0, 250.0),
    }
    return meta_map.get(res_class, (1920.0, 1080.0, 30.0, 250.0))

def export_onnx_binaries(out_dir=ONNX_OUT_DIR):
    os.makedirs(out_dir, exist_ok=True)

    # 1. Settings Optimizer model
    settings_meta_path = os.path.join(RUNS_DIR, "settings_optimizer", "model_meta_weights.json")
    with open(settings_meta_path) as fh:
        s_meta = json.load(fh)
    s_model = SettingsMLP()
    s_model.load_state_dict({k: torch.tensor(v) for k, v in s_meta["weights"].items()})
    s_model.eval()

    s_onnx = os.path.join(out_dir, "settings_model.onnx")
    torch.onnx.export(
        s_model, torch.zeros(1, 29), s_onnx,
        input_names=["clip_features"], output_names=["reward"], opset_version=17,
        dynamic_axes={"clip_features": {0: "batch"}, "reward": {0: "batch"}}
    )
    with open(os.path.join(out_dir, "settings_model_meta.json"), 'w') as fh:
        json.dump({"input_mean": s_meta["input_mean"], "input_std": s_meta["input_std"], "input_size": 29, "output_size": 1}, fh, indent=2)

    # 2. Track Predictor model
    track_meta_path = os.path.join(RUNS_DIR, "track_predictor", "model_meta_weights.json")
    with open(track_meta_path) as fh:
        t_meta = json.load(fh)
    t_model = TrackMLP()
    t_model.load_state_dict({k: torch.tensor(v) for k, v in t_meta["weights"].items()})
    t_model.eval()

    class TrackONNXWrapper(nn.Module):
        def __init__(self, base_model):
            super().__init__()
            self.base_model = base_model
        def forward(self, x):
            return torch.sigmoid(self.base_model(x))

    wrapped_t_model = TrackONNXWrapper(t_model)
    wrapped_t_model.eval()

    t_onnx = os.path.join(out_dir, "track_predictor.onnx")
    torch.onnx.export(
        wrapped_t_model, torch.zeros(1, 15), t_onnx,
        input_names=["track_features"], output_names=["survival_prob"], opset_version=17,
        dynamic_axes={"track_features": {0: "batch"}, "survival_prob": {0: "batch"}}
    )
    with open(os.path.join(out_dir, "track_predictor_meta.json"), 'w') as fh:
        json.dump({"input_mean": t_meta["input_mean"], "input_std": t_meta["input_std"], "input_size": 15, "output_size": 1}, fh, indent=2)

    # 3. NumPy JSON model with BatchNorm folding
    weights = t_meta["weights"]

    def fold_bn(linear_w, linear_b, bn_w, bn_b, bn_mean, bn_var, eps=1e-5):
        lw = np.array(linear_w, dtype=np.float32)
        lb = np.array(linear_b, dtype=np.float32)
        bw = np.array(bn_w, dtype=np.float32)
        bb = np.array(bn_b, dtype=np.float32)
        bm = np.array(bn_mean, dtype=np.float32)
        bv = np.array(bn_var, dtype=np.float32)

        scale = bw / np.sqrt(bv + eps)
        w_fold = lw * scale[:, np.newaxis]
        b_fold = (lb - bm) * scale + bb
        return w_fold.tolist(), b_fold.tolist()

    print("Folding BatchNorm layers for NumPy JSON model...")
    l1_w, l1_b = fold_bn(
        weights["network.0.weight"], weights["network.0.bias"],
        weights["network.1.weight"], weights["network.1.bias"],
        weights["network.1.running_mean"], weights["network.1.running_var"]
    )
    l2_w, l2_b = fold_bn(
        weights["network.4.weight"], weights["network.4.bias"],
        weights["network.5.weight"], weights["network.5.bias"],
        weights["network.5.running_mean"], weights["network.5.running_var"]
    )
    l3_w = weights["network.8.weight"]
    l3_b = weights["network.8.bias"]

    numpy_model = {
        "layer1_weight": l1_w,
        "layer1_bias": l1_b,
        "layer2_weight": l2_w,
        "layer2_bias": l2_b,
        "layer3_weight": l3_w,
        "layer3_bias": l3_b,
        "input_mean": t_meta["input_mean"],
        "input_std": t_meta["input_std"],
        "activation": "relu"
    }

    with open(os.path.join(RUNS_DIR, "track_predictor.json"), 'w') as fh:
        json.dump(numpy_model, fh, indent=4)

    print("ONNX and JSON models exported successfully!")

def export_defaults(model_meta_weights_path, recommended_defaults_json_path, defaults_json_path, region_weights_path=None):
    if not os.path.exists(model_meta_weights_path):
        print(f"Model file '{model_meta_weights_path}' not found. Cannot run presets grid search.")
        return
        
    with open(model_meta_weights_path, 'r') as f:
        meta = json.load(f)
        
    means = meta["input_mean"]
    stds = meta["input_std"]
    weights = meta["weights"]
    
    # Recreate PyTorch SettingsMLP and load weights for fast, reliable batch prediction
    model = SettingsMLP()
    model.load_state_dict({k: torch.tensor(v) for k, v in weights.items()})
    model.eval()
    if torch.cuda.is_available():
        model = model.cuda()
    
    # Define search parameter grid
    grid_patterns = [11, 15, 19, 25, 31, 41, 55]
    grid_searches = [51, 71, 91, 121, 181, 251]
    grid_correlations = [0.55, 0.65, 0.72, 0.80]
    grid_thresholds = [0.15, 0.25, 0.35, 0.45]
    grid_models = ['Loc', 'LocRot', 'Affine']
    
    # Build list of all candidate settings dictionaries (7*6*4*4*3 = 2,016 candidates)
    candidates = []
    for pattern in grid_patterns:
        for search in grid_searches:
            for corr in grid_correlations:
                for thresh in grid_thresholds:
                    for model_name in grid_models:
                        candidates.append({
                            "pattern_size": pattern,
                            "search_size": search,
                            "correlation": corr,
                            "threshold": thresh,
                            "motion_model": model_name
                        })
                        
    print(f"Grid search options per class: {len(candidates)}")
    recommendations = {}
    
    # Recommend optimal settings for each Resolution class + Footage type combo
    for res_class in RESOLUTION_CLASSES:
        width, height, fps, frame_count = get_resolution_metadata(res_class)
        recommendations[res_class] = {}
        
        for f_type in FOOTAGE_TYPES:
            features_batch = []
            for cand in candidates:
                # 1. Clip features (4)
                clip_feats = [width, height, fps, frame_count]
                # 2. Boolean switches (2)
                switches = [0.0, 0.0]
                # 3. Settings features (4)
                settings_feats = [
                    float(cand["pattern_size"]),
                    float(cand["search_size"]),
                    cand["correlation"],
                    cand["threshold"]
                ]
                # 4. One-hots (14)
                f_type_oh = get_one_hot(f_type, FOOTAGE_TYPES)
                m_model_oh = get_one_hot(cand["motion_model"], MOTION_MODELS)
                
                # 5. Video features (4)
                v_feats = [0.5, 0.0, 0.0, 0.003] # mean_motion, zoom_divergence, distortion_factor, noise_ratio
                if f_type == "ACTION":
                    v_feats[0] = 3.5
                elif f_type == "DRONE":
                    v_feats[1] = 0.5
                
                raw_vector = clip_feats + switches + settings_feats + f_type_oh + m_model_oh + v_feats
                # Normalize
                norm_vector = [(val - m) / s for val, m, s in zip(raw_vector, means, stds)]
                features_batch.append(norm_vector)
                
            # Run prediction using PyTorch SettingsMLP
            with torch.no_grad():
                features_tensor = torch.tensor(features_batch, dtype=torch.float32)
                if torch.cuda.is_available():
                    features_tensor = features_tensor.cuda()
                rewards = model(features_tensor).flatten().cpu().tolist()
            
            # Find candidate with max reward
            max_idx = rewards.index(max(rewards))
            best_settings = candidates[max_idx].copy()
            best_settings["expected_reward"] = round(rewards[max_idx], 4)
            recommendations[res_class][f_type] = best_settings
            
    # Output recommended_defaults.json
    with open(recommended_defaults_json_path, 'w') as f:
        json.dump(recommendations, f, indent=4)
    print(f"Recommended presets report generated at: {recommended_defaults_json_path}")
    
    # Compile a valid defaults.json for the addon presets
    defaults_model = {
        "version": 1,
        "description": "Pretrained model from community data - generated by defaults optimizer",
        "footage_classes": {},
        "footage_type_adjustments": {
            "DRONE": {
                "pattern_size_mult": 1.6,
                "search_size_mult": 1.8,
                "correlation_offset": -0.25,
                "threshold_mult": 0.27,
                "motion_model": "Affine"
            },
            "INDOOR": {
                "pattern_size_mult": 1.4,
                "search_size_mult": 1.5,
                "correlation_offset": -0.2,
                "threshold_mult": 0.27,
                "motion_model": "Affine"
            },
            "OUTDOOR": {
                "pattern_size_mult": 1.2,
                "search_size_mult": 1.3,
                "correlation_offset": -0.1,
                "threshold_mult": 0.8,
                "motion_model": "LocRot"
            },
            "HANDHELD": {
                "pattern_size_mult": 1.3,
                "search_size_mult": 1.6,
                "correlation_offset": -0.15,
                "threshold_mult": 0.6,
                "motion_model": "Affine"
            },
            "GIMBAL": {
                "pattern_size_mult": 1.0,
                "search_size_mult": 0.9,
                "correlation_offset": 0.05,
                "threshold_mult": 1.0,
                "motion_model": "LocRot"
            },
            "ACTION": {
                "pattern_size_mult": 1.5,
                "search_size_mult": 2.0,
                "correlation_offset": -0.25,
                "threshold_mult": 0.4,
                "motion_model": "Affine"
            },
            "VFX": {
                "pattern_size_mult": 1.1,
                "search_size_mult": 1.1,
                "correlation_offset": 0.0,
                "threshold_mult": 0.9,
                "motion_model": "LocRot"
            }
        },
        "region_models": {},
        "global_stats": {
            "total_sessions": len(RESOLUTION_CLASSES),
            "successful_sessions": len(RESOLUTION_CLASSES),
            "avg_success_rate": 1.0,
            "avg_best_error": 0.3
        }
    }
    
    # Fill in footage_classes based on search results
    for res_class in RESOLUTION_CLASSES:
        best_auto_settings = recommendations[res_class]["AUTO"].copy()
        best_auto_settings.pop("expected_reward", None)
        
        defaults_model["footage_classes"][res_class] = {
            "sample_count": 5,
            "success_count": 5,
            "avg_success_rate": 1.0,
            "best_settings": best_auto_settings,
            "best_error": 0.3
        }
        
    # Fill in region_models if region_weights is available
    if region_weights_path and os.path.exists(region_weights_path):
        try:
            with open(region_weights_path, 'r') as fh:
                rw = json.load(fh)
            auto_rw = rw.get("AUTO", {})
            for r_name, r_weight in auto_rw.items():
                defaults_model["region_models"][r_name] = {
                    "success_rate": r_weight,
                    "priority": "prioritize" if r_weight > 0.7 else "normal"
                }
        except Exception as e:
            print(f"Could not load region weights for defaults.json formatting: {e}")
            
    with open(defaults_json_path, 'w') as f:
        json.dump(defaults_model, f, indent=4)
    print(f"Format-compliant defaults presets saved to: {defaults_json_path}")

# Run Exporters
export_onnx_binaries()

export_defaults(
    model_meta_weights_path=os.path.join(RUNS_DIR, "settings_optimizer", "model_meta_weights.json"),
    recommended_defaults_json_path=os.path.join(RUNS_DIR, "recommended_defaults.json"),
    defaults_json_path=os.path.join(RUNS_DIR, "defaults.json"),
    region_weights_path=os.path.join(RUNS_DIR, "region_weights.json")
)


In [ ]:
# Outputs Verification

expected = [
    os.path.join(ONNX_OUT_DIR, "track_predictor.onnx"),
    os.path.join(ONNX_OUT_DIR, "track_predictor_meta.json"),
    os.path.join(ONNX_OUT_DIR, "settings_model.onnx"),
    os.path.join(ONNX_OUT_DIR, "settings_model_meta.json"),
    os.path.join(ONNX_OUT_DIR, "patch_rigidity.onnx"),
    os.path.join(ONNX_OUT_DIR, "patch_rigidity_meta.json"),
    os.path.join(RUNS_DIR, "track_predictor.json"),
    os.path.join(RUNS_DIR, "defaults.json"),
    os.path.join(RUNS_DIR, "region_weights.json"),
    os.path.join(RUNS_DIR, "recommended_defaults.json")
]


# ONNX model validation
import onnxruntime as ort

def verify_onnx(path: str, input_shape: list, input_name: str):
    if not os.path.exists(path):
        print(f"❌ Cannot verify missing model: {os.path.basename(path)}")
        return
    try:
        sess = ort.InferenceSession(path)
        dummy = np.zeros(input_shape, dtype=np.float32)
        out = sess.run(None, {input_name: dummy})
        print(f"✅ {os.path.basename(path)}: verified successfully (output shape {out[0].shape})")
    except Exception as e:
        print(f"❌ ONNX Verification failed for {os.path.basename(path)}: {e}")

print("ONNX Model Verification:")
verify_onnx(os.path.join(ONNX_OUT_DIR, "settings_model.onnx"), [1, 29], "clip_features")
verify_onnx(os.path.join(ONNX_OUT_DIR, "track_predictor.onnx"), [1, 20], "track_features")
verify_onnx(os.path.join(ONNX_OUT_DIR, "patch_rigidity.onnx"), [1, 1, 32, 32], "patch_pixels")
print()

print("Trained Model Output Verification:\n")
for f in expected:
    exists = os.path.exists(f)
    size   = os.path.getsize(f) // 1024 if exists else 0
    status = f'✅  {size:4d} KB' if exists else '❌  MISSING'
    print(f'{status}   {os.path.basename(f)}  ({f})')


In [ ]:
# Downloader Handler for Colab
try:
    from google.colab import files
    import zipfile
    
    zip_path = "/content/autosolve_models.zip"
    print("Packaging trained models and presets into zip file...")
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for f in expected:
            if os.path.exists(f):
                zipf.write(f, os.path.basename(f))
                print(f"  Added to zip: {os.path.basename(f)}")
                
    if os.path.exists(zip_path) and os.path.getsize(zip_path) > 0:
        print(f"\nDownloading package: autosolve_models.zip ({os.path.getsize(zip_path)//1024} KB)")
        files.download(zip_path)
    else:
        print("No outputs found to pack.")
except ImportError:
    print("Running locally. Models are located in your local project 'ml/runs/' folder.")
